# Martini: Membrane Thickness Optimization

This notebook optimizes Martini M2 force-field parameters to match a target membrane thickness via DiffTRe, using GROMACS as the simulation engine.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook demonstrates optimization of membrane thickness — a key structural property of lipid bilayers. The `MembraneThickness` observable measures the distance between PO4 head-group beads across the bilayer, and DiffTRe computes gradients to drive the simulated thickness toward the experimental target.

## Imports

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import MDAnalysis
import optax
from mythos.energy.base import ComposedEnergyFunction
from mythos.energy.martini.base import MartiniTopology
from mythos.energy.martini.m2 import (
    LJ,
    Angle,
    AngleConfiguration,
    Bond,
    BondConfiguration,
    LJConfiguration,
)
from mythos.input.gromacs_input import read_params_from_topology
from mythos.observables.membrane_thickness import MembraneThickness
from mythos.optimization.objective import DiffTReObjective
from mythos.optimization.optimization import RayOptimizer
from mythos.simulators.gromacs.gromacs import GromacsSimulator
from mythos.simulators.gromacs.utils import preprocess_topology
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)

## Configuration

In [ ]:
INPUT_DIR = Path("../../data/templates/martini/m2/DMPC/273K")
TARGET_THICKNESS = 3.5  # nm — adjust to your target
NUM_SIMS = 1
OPT_STEPS = 100
LEARNING_RATE = 5e-4
TEMPERATURE = 273.0
EQUILIBRATION_STEPS = 200_000
SIMULATION_STEPS = 500_000
SNAPSHOT_STEPS = 10_000
GROMACS_BINARY = None

## Preprocess topology and build energy function

To build energy functions for martini we need to read parameters from a
preprocessed topology file (which expands any macros and includes all files).
Likewise we need to read the compiled topology which comes in the .tpr format
read into a mythose `MartiniTopology` object via MDAnalysis.

In [ ]:
preprocessed_top = INPUT_DIR / "preprocessed.top"
preprocessed_tpr = INPUT_DIR / "preprocessed.tpr"

if not preprocessed_top.exists() or not preprocessed_tpr.exists():
    preprocess_topology(INPUT_DIR, gromacs_binary=GROMACS_BINARY)

universe = MDAnalysis.Universe(preprocessed_tpr)
universe.transfer_to_memory()
parameters = read_params_from_topology(preprocessed_top)

top = MartiniTopology.from_universe(universe)

energy_fn = ComposedEnergyFunction(energy_fns=[
    LJ.from_topology(topology=top, params=LJConfiguration(**parameters["nonbond_params"])),
    Bond.from_topology(topology=top, params=BondConfiguration(**parameters["bond_params"])),
    Angle.from_topology(topology=top, params=AngleConfiguration(**parameters["angle_params"])),
])

## Create simulators, observable, and objective

In [ ]:
simulator = GromacsSimulator.create_n(
    NUM_SIMS,
    input_dir=INPUT_DIR,
    energy_fn=energy_fn.with_props(unbonded_neighbors=None),
    equilibration_steps=EQUILIBRATION_STEPS,
    simulation_steps=SIMULATION_STEPS,
    input_overrides={"nstxout": SNAPSHOT_STEPS, "ref-t": TEMPERATURE},
    binary_path=GROMACS_BINARY,
)

thickness_obs = MembraneThickness(
    topology=universe,
    lipid_sel="name GL1 GL2",
    thickness_sel="name PO4",
)


def thickness_loss(traj, weights, *_):
    all_thickness = thickness_obs(traj)
    expected_thickness = jnp.dot(weights, all_thickness)
    loss = jnp.sqrt((TARGET_THICKNESS - expected_thickness) ** 2)
    return loss, (("thickness", expected_thickness), ())


thickness_obj = DiffTReObjective(
    energy_fn=energy_fn,
    grad_or_loss_fn=thickness_loss,
    required_observables=[i for s in simulator for i in s.exposes()],
    name="MembraneThickness",
)

## Run the optimization

In [ ]:
opt = RayOptimizer(
    simulators=simulator,
    objectives=[thickness_obj],
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=lambda x: x[0],
    logger=ConsoleLogger(),
)

opt_params = energy_fn.opt_params()

print("\n=== Setup Complete ===")
print(f"Data directory: {INPUT_DIR}")
print(f"Energy function terms: {[type(fn).__name__ for fn in energy_fn.energy_fns]}")
print(f"Target thickness: {TARGET_THICKNESS} nm")
print(f"Optimizing {len(opt_params)} parameters")

output = opt.run(params=opt_params, n_steps=OPT_STEPS)
print("\nOptimization complete!")